## TDFP

Top-down forecast-proportions **order reconciliation** using the cleaned simulation pipeline.

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import TopDown

from inventory_pipeline import save_pickle, run_fixed_loop
#from inventory_pipeline import dual_save, run_fixed_loop

In [2]:
lead_time = 1
MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
on = ['ot_90', 'ot_95', 'ot_99']

lp = {"l10": (0, 3049), "l11": (3049, 3049 + 9147), "l12": (3049 + 9147, 3049 + 9147 + 30490)}
lvls = []
for k, (a, b) in lp.items():
    lvls.append([k[1:]] * 28 * (b - a))
levels = pd.concat([pd.DataFrame({'levels': lvls[i]}) for i in range(len(lvls))], ignore_index=True)

test = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)
truth = test['y'].reset_index(drop=True)
S = pd.read_pickle(f"{path1}721S_df.pkl")
tags = pd.read_pickle(f"{path1}tags.bin")

ts = pd.concat([levels, test[['unique_id', 'ds']].reset_index(drop=True)], axis=1)
ts2 = ts[ts['levels'].isin(['10', '11', '12'])].reset_index(drop=True)

s = S.copy()
l10_12 = [
    'total/cat_id/dept_id/item_id',
    'total/state_id/cat_id/dept_id/item_id',
    'total/state_id/store_id/cat_id/dept_id/item_id',
]
prods = [u.split('/')[-1] for u in tags[l10_12[0]]]

lgbs = pd.read_pickle(f"lgbInvtSim_L{lead_time}.pkl").reset_index(drop=True)
etss = pd.read_pickle(f"etsInvtSim_L{lead_time}.pkl").reset_index(drop=True)
base_all = pd.concat([lgbs, etss], ignore_index=True)

TDFP = HierarchicalReconciliation(reconcilers=[TopDown(method='forecast_proportions')])

In [3]:
prods

['FOODS_1_001',
 'FOODS_1_002',
 'FOODS_1_003',
 'FOODS_1_004',
 'FOODS_1_005',
 'FOODS_1_006',
 'FOODS_1_008',
 'FOODS_1_009',
 'FOODS_1_010',
 'FOODS_1_011',
 'FOODS_1_012',
 'FOODS_1_013',
 'FOODS_1_014',
 'FOODS_1_015',
 'FOODS_1_016',
 'FOODS_1_017',
 'FOODS_1_018',
 'FOODS_1_019',
 'FOODS_1_020',
 'FOODS_1_021',
 'FOODS_1_022',
 'FOODS_1_023',
 'FOODS_1_024',
 'FOODS_1_025',
 'FOODS_1_026',
 'FOODS_1_027',
 'FOODS_1_028',
 'FOODS_1_029',
 'FOODS_1_030',
 'FOODS_1_031',
 'FOODS_1_032',
 'FOODS_1_033',
 'FOODS_1_034',
 'FOODS_1_035',
 'FOODS_1_036',
 'FOODS_1_037',
 'FOODS_1_038',
 'FOODS_1_039',
 'FOODS_1_040',
 'FOODS_1_041',
 'FOODS_1_042',
 'FOODS_1_043',
 'FOODS_1_044',
 'FOODS_1_045',
 'FOODS_1_046',
 'FOODS_1_047',
 'FOODS_1_048',
 'FOODS_1_049',
 'FOODS_1_050',
 'FOODS_1_051',
 'FOODS_1_052',
 'FOODS_1_053',
 'FOODS_1_054',
 'FOODS_1_055',
 'FOODS_1_056',
 'FOODS_1_057',
 'FOODS_1_058',
 'FOODS_1_059',
 'FOODS_1_060',
 'FOODS_1_061',
 'FOODS_1_062',
 'FOODS_1_063',
 'FOODS_

In [4]:
OrderTd = []
for name in MODEL_NAMES:
    print(f"------------------------{name} Start!------------------------")
    df_name_orders = base_all.loc[base_all['name'] == name, on].reset_index(drop=True)
    df_name = pd.concat([ts2[['unique_id', 'ds']].reset_index(drop=True), df_name_orders], axis=1)

    per_prod = []
    for prod in tqdm(prods, leave=False):
        b = df_name[df_name['unique_id'].str.contains(prod)].copy()
        b['_row'] = b.index

        bs = s[s['unique_id'].str.contains(prod)].copy().reset_index(drop=True)
        bss = pd.concat(
            [bs[['unique_id']], bs.loc[:, bs.columns.str.contains(prod)].reset_index(drop=True)],
            axis=1,
        )

        uid_list = bs['unique_id'].tolist()
        btags = {
            l10_12[0]: [u for u in uid_list if len(u.split('/')) == 4],
            l10_12[1]: [u for u in uid_list if len(u.split('/')) == 5],
            l10_12[2]: [u for u in uid_list if len(u.split('/')) == 6],
        }

        rec = TDFP.reconcile(
            Y_hat_df=b[['unique_id', 'ds', 'ot_90', 'ot_95', 'ot_99']],
            S=bss,
            tags=btags,
        )[['unique_id', 'ds', 'ot_90/TopDown_method-forecast_proportions', 
             'ot_95/TopDown_method-forecast_proportions', 'ot_99/TopDown_method-forecast_proportions']]

        rec = rec.merge(b[['unique_id', 'ds', '_row']], on=['unique_id', 'ds'], how='left').sort_values('_row')
        per_prod.append(rec)

    rec_name = pd.concat(per_prod, ignore_index=True).sort_values('_row').drop(columns='_row').reset_index(drop=True)
    rec_name.insert(0, 'name', name)
    OrderTd.append(rec_name)

OrderTDFP = pd.concat(OrderTd, ignore_index=True)


------------------------lgb_base Start!------------------------


------------------------lgb_bu Start!------------------------


------------------------lgb_td Start!------------------------


------------------------lgb_mint Start!------------------------


------------------------ets_base Start!------------------------


------------------------ets_bu Start!------------------------


------------------------ets_td Start!------------------------


------------------------ets_mint Start!------------------------


In [5]:
OrderTDFP.columns = ['name', 'unique_id', 'ds']+on
OrderTDFP2 = OrderTDFP[['name']+on]
save_pickle(OrderTDFP2, f"OrderTDFP_L{lead_time}.pkl")
OrderTDFP2.head()

,name,ot_90,ot_95,ot_99
0,lgb_base,5.648312,7.249530,10.253148
1,lgb_base,2.989336,2.989336,2.989336
2,lgb_base,5.119798,5.119798,5.119798
3,lgb_base,8.181289,8.181289,8.181289
4,lgb_base,0.068346,0.068346,0.068346


In [6]:
Invt_df = []
OrderTDFP = OrderTDFP2.copy()
for name in MODEL_NAMES:
    base_ref = base_all.loc[base_all['name'] == name].reset_index(drop=True)
    order_ref = OrderTDFP.loc[OrderTDFP['name'] == name].reset_index(drop=True)

    fixed_orders = pd.concat(
        [
            base_ref[['forecasts', 'sst_90', 'sst_95', 'sst_99']].reset_index(drop=True),
            order_ref[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    df = run_fixed_loop(
        truth=truth,
        forecast_source=base_ref['forecasts'].reset_index(drop=True),
        fixed_orders=fixed_orders,
        NAME=name,
        L_=lead_time,
    )
    Invt_df.append(df)

TDFPOrder = pd.concat(Invt_df, ignore_index=True)
save_pickle(TDFPOrder, f"TDFPOrder_L{lead_time}.pkl")
TDFPOrder.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:13<00:00, 582.91it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,lgb_base,4.0,5.689794,5.648312,0.000000,5.648312,5.648312,7.338106,1.689794,0.000000,...,0.000000,10.253148,0.000000,10.253148,10.253148,11.942942,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,5.648312,2.989336,2.989336,5.327442,2.338106,0.000000,...,0.000000,10.253148,10.253148,2.989336,2.989336,9.932278,6.942942,0.0,6.942942,0.0
2,lgb_base,7.0,4.798928,5.648312,2.989336,5.119798,5.119798,3.447240,-1.672558,1.672558,...,1.355461,10.253148,2.989336,5.119798,5.119798,8.052076,2.932278,0.0,2.932278,0.0
3,lgb_base,1.0,5.980217,5.648312,5.119798,8.181289,8.181289,10.628529,2.447240,0.000000,...,0.000000,10.253148,5.119798,8.181289,8.181289,15.233366,7.052076,0.0,7.052076,0.0
4,lgb_base,9.0,5.048564,5.648312,8.181289,0.068346,0.068346,1.696875,1.628529,0.000000,...,0.000000,10.253148,8.181289,0.068346,0.068346,6.301712,6.233366,0.0,6.233366,0.0
